In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 

In [ ]:
import sys

import os
import spotiphy
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import scanpy as sc
import torch
import cv2

bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
2026-09-11 17:18:13.912095: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-11 17:18:14.228937: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-11 17:18:14.368703: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to

In [3]:
from scipy.io import mmread
import anndata
import pandas as pd
import numpy as np
import scanpy as sc

In [13]:
os.chdir('/data_hdd4/yuseong/cancer_aging/analysis/h.github/')

In [16]:
adata = sc.read_h5ad('Fig5/data/5j/patient_all_processed.h5ad')

In [15]:
adata_vis2 = sc.read_h5ad('Fig5/data/5j/Post_visium_processed.h5ad')

In [17]:
adata_sc = adata.copy()
adata_st = adata_vis2.copy()

In [18]:
key_type = "celltype_f"
type_list = sorted(list(adata_sc.obs[key_type].unique().astype(str)))
print(f"There are {len(type_list)} cell types: {type_list}")
adata_st.var_names_make_unique()

There are 11 cell types: ['Bcell', 'Endothelial', 'Fibroblast', 'MuscleCell', 'Myeloid', 'PlasmaCell', 'T.NKcell', 'malignant', 'non_malignant', 'sen_endothelial', 'sen_malignant']


In [19]:
adata_st.obsm["spatial"] = adata_st.obsm["spatial"].astype(np.int32)
# print(adata_st.obsm['spatial'])

In [ ]:
### decomposition

In [20]:
rctd = pd.read_csv('Fig5/data/5d-f/Post_RCTD_prop_normalized.csv', index_col=0)
rctd.head()

,Bcell,Endothelial,Fibroblast,Malignant,MuscleCell,Myeloid,Non_Malignant,PlasmaCell,Sen_Endothelial,Sen_Malignant,T.NKcell
AACACTTGGCAAGGAA-1,0.000042,0.000042,0.029902,0.158657,0.000042,0.000042,0.640645,0.011327,0.146228,0.013029,0.000042
AACAGGATTCATAGTT-1,0.018789,0.080819,0.708528,0.042486,0.000040,0.018800,0.038581,0.046063,0.000077,0.034001,0.011816
AACAGGTTCACCGAAG-1,0.019825,0.169286,0.741542,0.017310,0.008319,0.005052,0.006983,0.026262,0.005222,0.000120,0.000079
AACATAGTCTATCTAC-1,0.000128,0.021733,0.485910,0.000128,0.002859,0.154043,0.002639,0.018177,0.024994,0.193587,0.095803
AACCAGAATCAGACGT-1,0.000040,0.000040,0.000040,0.406073,0.000040,0.000040,0.569842,0.000040,0.000040,0.000040,0.023766


In [21]:
type_list2 = ['Sen_Malignant' if x == 'sen_malignant' else ('Sen_Endothelial' if x == 'sen_endothelial' 
                                                           else ('Malignant' if x == 'malignant' 
                                                                 else (x =='Malignant' if x == 'malignant' else ('Non_Malignant' if x == 'non_malignant' else x)))) for x in type_list]
rctd = rctd[type_list2]
rctd.head()

,Bcell,Endothelial,Fibroblast,MuscleCell,Myeloid,PlasmaCell,T.NKcell,Malignant,Non_Malignant,Sen_Endothelial,Sen_Malignant
AACACTTGGCAAGGAA-1,0.000042,0.000042,0.029902,0.000042,0.000042,0.011327,0.000042,0.158657,0.640645,0.146228,0.013029
AACAGGATTCATAGTT-1,0.018789,0.080819,0.708528,0.000040,0.018800,0.046063,0.011816,0.042486,0.038581,0.000077,0.034001
AACAGGTTCACCGAAG-1,0.019825,0.169286,0.741542,0.008319,0.005052,0.026262,0.000079,0.017310,0.006983,0.005222,0.000120
AACATAGTCTATCTAC-1,0.000128,0.021733,0.485910,0.002859,0.154043,0.018177,0.095803,0.000128,0.002639,0.024994,0.193587
AACCAGAATCAGACGT-1,0.000040,0.000040,0.000040,0.000040,0.000040,0.000040,0.023766,0.406073,0.569842,0.000040,0.000040


In [ ]:
### run spotiphy

In [23]:
adata_st.var_names_make_unique()

In [ ]:
n_cell_df = pd.DataFrame({'cell_num': 5}, index=range(adata_st.shape[0]))
n_cell_df.shape
n_cell = n_cell_df["cell_count"].values
adata_st_decomposed = spotiphy.deconvolution.decomposition(
    adata_st,
    adata_sc,
    key_type,
    np.array(rctd),
    save=True,
    out_dir='Fig5/data/5j/decomp',
    verbose=1,
    spot_location=adata_st.obsm["spatial"],
    filtering_gene=True,
    n_cell= n_cell, 
    filename="ST_decomposition_5.h5ad",
)

In [ ]:
adata_st_decomposed.obs.cell_type.value_counts()

cell_type
Non_Malignant      3204
Fibroblast         1567
Malignant           902
Endothelial         537
Sen_Malignant       427
Sen_Endothelial     228
PlasmaCell          164
Myeloid              82
T.NKcell             54
Bcell                 9
MuscleCell            6
Name: count, dtype: int64